In [2]:
# load module
import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
from sklearn.preprocessing import StandardScaler

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

import tensorflow as tf 
import tensorboard as tb 
tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']

/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(


In [15]:
def data_processing(path = '../../sample_table/ewma_6h_scaling.csv'):
    data = pd.read_csv(path)
    columns = ['REG_DTIME', 'h_dong', 'count', 'pops',
       'windspd', 'humid', 'temp', 'precip_form', 'precip', 'isHoliday']
    data = data[columns]
    data['REG_DTIME'] = pd.to_datetime(data['REG_DTIME'])
    data['DOW'] = data['REG_DTIME'].dt.dayofweek
    data['HOD'] = data['REG_DTIME'].dt.hour
    data["time_idx"] =  \
    (data["REG_DTIME"].dt.month) * data["REG_DTIME"].dt.daysinmonth * 24  + \
    data["REG_DTIME"].dt.day * 24  + \
    data["REG_DTIME"].dt.hour 
    data["time_idx"] -= data["time_idx"].min()
    data['DOW'] = data['DOW'].astype(str)
    data['HOD'] = data['HOD'].astype(str)
    data['precip_form'] = data['precip_form'].astype(str)
    data['isHoliday'] = data['isHoliday'].astype(str)
    #data['count'] = data['count'].round(5)
    return data

# functions
def moving_average_com(df, unit):
    '''
    training datset에 ewma를 적용하는 함수 
    '''
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    # forward df
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        inv_dong_df = df[df['h_dong'] == dong][::-1]
        dong_df['count'] = dong_df['count'].ewm(com = unit).mean()
        for_ewma = dong_df['count'].ewm(com = unit).mean()
        back_ewma = inv_dong_df['count'].ewm(com = unit).mean()

        ewma = (for_ewma + back_ewma)/2 
        df['count'][dong_df.index] = ewma 
    df['count'] = df['count']  /  df['count'].max() * (max_value+2)
    
    #print(max_value , df['count'].max())
    return df


def moving_average_span(df, unit):
    '''
    training datset에 ewma를 적용하는 함수 
    '''
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    # forward df
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        inv_dong_df = df[df['h_dong'] == dong][::-1]
        dong_df['count'] = dong_df['count'].ewm(span = unit).mean()
        for_ewma = dong_df['count'].ewm(span = unit).mean()
        back_ewma = inv_dong_df['count'].ewm(span = unit).mean()

        ewma = (for_ewma + back_ewma)/2 
        df['count'][dong_df.index] = ewma 
    df['count'] = df['count']  /  df['count'].max() * (max_value+2)
    
    #print(max_value , df['count'].max())
    return df

def moving_average_halflife(df, unit):
    '''
    training datset에 ewma를 적용하는 함수 
    '''
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    # forward df
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        inv_dong_df = df[df['h_dong'] == dong][::-1]
        dong_df['count'] = dong_df['count'].ewm(halflife = unit).mean()
        for_ewma = dong_df['count'].ewm(halflife = unit).mean()
        back_ewma = inv_dong_df['count'].ewm(halflife = unit).mean()

        ewma = (for_ewma + back_ewma)/2 
        df['count'][dong_df.index] = ewma 
    df['count'] = df['count']  /  df['count'].max() * (max_value+2)
    
    #print(max_value , df['count'].max())
    return df


def moving_average_alpha(df, unit):
    '''
    training datset에 ewma를 적용하는 함수 
    '''
    ret_df = pd.DataFrame()
    max_value = df['count'].max()
    # forward df
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        inv_dong_df = df[df['h_dong'] == dong][::-1]
        dong_df['count'] = dong_df['count'].ewm(alpha = unit).mean()
        for_ewma = dong_df['count'].ewm(alpha = unit).mean()
        back_ewma = inv_dong_df['count'].ewm(alpha = unit).mean()

        ewma = (for_ewma + back_ewma)/2 
        df['count'][dong_df.index] = ewma 
    df['count'] = df['count']  /  df['count'].max() * (max_value)
    
    #print(max_value , df['count'].max())
    return df


def dong_actual(start , end , ewma , option):
    dt = pd.date_range(start, end , freq = '1h').astype(str)

    for dong in dongs[14:15]:
        plt.figure(figsize=(12,4))
        dong_df = mv_data[mv_data['h_dong'] == dong]
        time_df = dong_df[dong_df['REG_DTIME'].isin(dt)]
        time_df.reset_index(inplace=True)
        plt.scatter(time_df.index, time_df['count'] , label=f'{dong} ewma' , s= 1 ,alpha=0.5 )

        org_dong_df = org_data[org_data['h_dong'] == dong]
        org_time_df = org_dong_df[org_dong_df['REG_DTIME'].isin(dt)]
        org_time_df.reset_index(inplace=True)
        plt.scatter(org_time_df.index, org_time_df['count'] , label=f'{dong} orignal' , s= 1, alpha = 1)

        
        plt.title(f'{dong} actual & {dong} , {option}')
        plt.legend()

def get_training(data , max_prediction_length, max_encoder_length):
    # traing data 생성
    max_prediction_length = max_prediction_length
    max_encoder_length = max_encoder_length
    training_cutoff = data["time_idx"].max() - max_prediction_length

    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = "count",
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=24,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=[],
        time_varying_known_reals=[],
        #variable_groups={"special_days": special_days},  # group of categorical variables can be treated as one variable
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=['count'], 
        target_normalizer=GroupNormalizer(
            groups=["h_dong"], 
            transformation="relu",
            center = False
        ),
        add_relative_time_idx=True,
        add_target_scales=False,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True)
    return training


In [14]:
data = data_processing()
data

,REG_DTIME,h_dong,count,pops,windspd,humid,temp,precip_form,precip,isHoliday,DOW,HOD,time_idx
0,2021-01-01 00:00:00,동산면,0.0,1478,0.4,76.0,-9.3,0.0,0.0,True,4,0,0
1,2021-01-01 00:00:00,후평1동,0.0,11827,0.8,57.0,-7.6,0.0,0.0,True,4,0,0
2,2021-01-01 00:00:00,사북면,0.0,2551,0.7,66.0,-10.3,0.0,0.0,True,4,0,0
3,2021-01-01 00:00:00,신북읍,0.0,7694,0.4,59.0,-8.1,0.0,0.0,True,4,0,0
4,2021-01-01 00:00:00,석사동,0.0,35412,0.8,57.0,-7.6,0.0,0.0,True,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
183955,2021-12-31 23:00:00,근화동,0.0,8540,0.7,72.0,-12.4,0.0,0.0,False,4,23,8927
183956,2021-12-31 23:00:00,동 면,0.0,19649,0.7,72.0,-12.4,0.0,0.0,False,4,23,8927
183957,2021-12-31 23:00:00,신사우동,0.0,22935,0.7,72.0,-12.4,0.0,0.0,False,4,23,8927
183958,2021-12-31 23:00:00,약사명동,0.0,3278,0.7,72.0,-12.4,0.0,0.0,False,4,23,8927


In [16]:
class params_setting():
    def __init___(org_data_path):
        self.data = data_processing(org_data_path)
    
    def ewma(data , method ,unit , nei):
        data_meta = {}
        data_meta["emwa"] = unit
        data_meta["nei"] = nei
        
        
    